In [1]:
%load_ext autoreload
%autoreload 2

In [87]:
import numpy as np
from cccaatl_ml.core.tensor import Tensor 
from cccaatl_ml.core.layer import *
from cccaatl_ml.core.activations import *

import time
from functools import partial

In [91]:
def benchmark(func, warmup:int = 10):
    times = []
    for _ in range(warmup): 
        start = time.perf_counter()
        func()
        end = time.perf_counter()
        times.append(end - start)
    return sorted(times)[warmup // 2]

In [51]:
a = np.random.rand(10, 1, 2)
b = np.random.rand(50, 1)
a_tensor = Tensor(a)
b_tensor = Tensor(b)

# testing pairwise arithmetic and broadcasting
assert np.allclose(a + b, (a_tensor + b_tensor)._array)
assert np.allclose(a * b, (a_tensor * b_tensor)._array)
assert np.allclose(a - b, (a_tensor - b_tensor)._array)
assert np.allclose(a / b, (a_tensor / b_tensor)._array)

# testing reductions 
assert np.equal(b.sum(), (b_tensor.sum())._array)
assert np.equal(b.mean(), (b_tensor.mean())._array)
assert np.equal(b.max(), (b_tensor.max())._array)

In [28]:
A = np.random.rand(10, 50, 30)
B = np.random.rand(30, 20)
A_tensor = Tensor(A)
B_tensor = Tensor(B)

# testing matrix operations
assert np.allclose(A @ B, (A_tensor @ B_tensor)._array)
assert np.allclose(A.reshape(500, -1), (A_tensor.reshape(500, -1))._array)
assert np.allclose(B.transpose(), (B_tensor.transpose())._array)

In [56]:
relu = ReLU()
sigmoid = Sigmoid() 
tanh = Tanh() 
softmax = Softmax()

c = Tensor([-2, -1, 0, 1, 2])
d = Tensor(np.random.rand(1000))

# testing activations
assert np.allclose(relu(c)._array, np.array([0, 0, 0, 1, 2]))
assert softmax(c).sum()._array == np.array([1.0])
assert tanh(d).min()._array > np.array([-1]) and tanh(d).max()._array < np.array([1])
assert sigmoid(d).min()._array > np.array([0]) and sigmoid(d).max()._array < np.array([1])

In [93]:
# testing forward pass with batch broadcasting
net = Sequential([
    Linear(784, 100),
    ReLU(),
    Linear(100, 100),
    ReLU(),
    Linear(100, 10)
])

dummy_batch = Tensor(np.random.rand(32, 784))

out = net(dummy_batch)

assert out.shape == (32, 10)

In [95]:
print(f"Median forward pass time: {benchmark(partial(net, dummy_batch))}")

Median forward pass time: 0.00017453999316785485
